<a href="https://colab.research.google.com/github/Chikka-Pradhayani/ABTalks-60-Days-AI-Challenge/blob/main/Day-33.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langsmith langchain langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 23.3 MB/s eta 0:00:00


In [2]:
import os
import json
import time
import uuid
import logging
from datetime import datetime

from langsmith import Client

In [3]:
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = "your_langsmith_api_key"
os.environ["LANGSMITH_PROJECT"] = "day-33-ai-debugging"

In [5]:
client = Client()

print("LangSmith connection successful")
print("Project:", os.environ["LANGSMITH_PROJECT"])

LangSmith connection successful
Project: day-33-ai-debugging


In [7]:
%whos

Variable   Type      Data/Info
------------------------------
Client     type      <class 'langsmith.client.Client'>
client     Client    Client (API URL: https://api.smith.langchain.com)
datetime   type      <class 'datetime.datetime'>
json       module    <module 'json' from '/usr<...>on3.13/json/__init__.py'>
logging    module    <module 'logging' from '/<...>.13/logging/__init__.py'>
os         module    <module 'os' (frozen)>
time       module    <module 'time' (built-in)>
uuid       module    <module 'uuid' from '/usr<...>/lib/python3.13/uuid.py'>


In [8]:
evaluation_results = []
print("Evaluation results container created")

Evaluation results container created


In [9]:
%whos

Variable             Type      Data/Info
----------------------------------------
Client               type      <class 'langsmith.client.Client'>
client               Client    Client (API URL: https://api.smith.langchain.com)
datetime             type      <class 'datetime.datetime'>
evaluation_results   list      n=0
json                 module    <module 'json' from '/usr<...>on3.13/json/__init__.py'>
logging              module    <module 'logging' from '/<...>.13/logging/__init__.py'>
os                   module    <module 'os' (frozen)>
time                 module    <module 'time' (built-in)>
uuid                 module    <module 'uuid' from '/usr<...>/lib/python3.13/uuid.py'>


In [22]:
!pip install -q -U transformers sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 96.5 MB/s eta 0:00:00


In [23]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Local model loaded successfully")

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Local model loaded successfully


In [24]:
question = "Explain retrieval augmented generation in one sentence."

inputs = tokenizer(
    question,
    return_tensors="pt"
)

outputs = model.generate(
    **inputs,
    max_new_tokens=50
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(answer)

augmented generation of augmented generation


In [25]:
def local_generate(context, question):
    prompt = f"""
Answer the question using only the provided context.

Context:
{context}

Question:
{question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [26]:
def rag_pipeline(query):
    retrieved_docs = retrieve(query)

    context = "\n\n".join(
        doc["content"] for doc in retrieved_docs
    )

    answer = local_generate(
        context,
        query
    )

    return {
        "query": query,
        "retrieved_docs": retrieved_docs,
        "context": context,
        "answer": answer
    }

In [27]:
result = rag_pipeline(
    "What is retrieval augmented generation?"
)

print("Query:", result["query"])

print("\nRetrieved documents:")

for doc in result["retrieved_docs"]:
    print("-", doc["id"])

print("\nAnswer:")

print(result["answer"])

Query: What is retrieval augmented generation?

Retrieved documents:
- doc_1
- doc_3

Answer:
combines information retrieval with language generation


In [28]:
def bad_retrieve(query, k=2):
    return [
        documents[1],
        documents[2]
    ]

In [29]:
def failure_1_pipeline(query):
    retrieved_docs = bad_retrieve(query)

    context = "\n\n".join(
        doc["content"] for doc in retrieved_docs
    )

    answer = local_generate(
        context,
        query
    )

    return {
        "query": query,
        "retrieved_docs": retrieved_docs,
        "context": context,
        "answer": answer
    }

In [30]:
failure_1 = failure_1_pipeline(
    "What is retrieval augmented generation?"
)

print("Query:", failure_1["query"])

print("\nRetrieved documents:")

for doc in failure_1["retrieved_docs"]:
    print("-", doc["id"])

print("\nAnswer:")
print(failure_1["answer"])

Query: What is retrieval augmented generation?

Retrieved documents:
- doc_2
- doc_3

Answer:
Good chunking helps a RAG system retrieve relevant information efficiently and reduces irrelevant context


In [31]:
def failure_2_pipeline(query):
    retrieved_docs = retrieve(query)

    context = "\n\n".join(
        doc["content"] for doc in retrieved_docs
    )

    bad_prompt = f"""
Answer the question from your own knowledge.

Question:
{query}

Additional information:
{context}
"""

    inputs = tokenizer(
        bad_prompt,
        return_tensors="pt",
        truncation=True
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return {
        "query": query,
        "retrieved_docs": retrieved_docs,
        "context": context,
        "answer": answer
    }

In [32]:
failure_2 = failure_2_pipeline(
    "What is hallucination in a large language model?"
)

print("Query:", failure_2["query"])

print("\nRetrieved documents:")

for doc in failure_2["retrieved_docs"]:
    print("-", doc["id"])

print("\nAnswer:")
print(failure_2["answer"])

Query: What is hallucination in a large language model?

Retrieved documents:
- doc_3
- doc_1

Answer:
false, unsupported, or not grounded in the available evidence


In [33]:
def failure_3_pipeline(query):
    retrieved_docs = retrieve(query)

    context = "\n\n".join(
        doc["content"] for doc in retrieved_docs
    )

    drift_prompt = f"""
Give a detailed answer to this question.

Context:
{context}

Question:
{query}

Include additional useful information even if it is not explicitly present in the context.
"""

    inputs = tokenizer(
        drift_prompt,
        return_tensors="pt",
        truncation=True
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=150
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return {
        "query": query,
        "retrieved_docs": retrieved_docs,
        "context": context,
        "answer": answer
    }

In [34]:
failure_3 = failure_3_pipeline(
    "What is chunking important for in a RAG system?"
)

print("Query:", failure_3["query"])

print("\nRetrieved documents:")

for doc in failure_3["retrieved_docs"]:
    print("-", doc["id"])

print("\nAnswer:")
print(failure_3["answer"])

Query: What is chunking important for in a RAG system?

Retrieved documents:
- doc_2
- doc_3

Answer:
retrieve relevant information efficiently and reduces irrelevant context


In [35]:
def isolate_components(query, retriever=retrieve):
    print("=" * 60)
    print("QUERY")
    print("=" * 60)
    print(query)

    print("\n" + "=" * 60)
    print("STEP 1 — RETRIEVAL")
    print("=" * 60)

    retrieved_docs = retriever(query)

    for doc in retrieved_docs:
        print(f"[{doc['id']}] {doc['content']}")

    print("\n" + "=" * 60)
    print("STEP 2 — PROMPT CONSTRUCTION")
    print("=" * 60)

    context = "\n\n".join(
        doc["content"] for doc in retrieved_docs
    )

    prompt_text = f"""
Answer the question using only the provided context.

Context:
{context}

Question:
{query}

Answer:
"""

    print(prompt_text)

    print("\n" + "=" * 60)
    print("STEP 3 — GENERATION")
    print("=" * 60)

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print(answer)

    return {
        "query": query,
        "retrieved_docs": retrieved_docs,
        "prompt": prompt_text,
        "answer": answer
    }

In [36]:
isolation_result = isolate_components(
    "What is retrieval augmented generation?"
)

QUERY
What is retrieval augmented generation?

STEP 1 — RETRIEVAL
[doc_1] Retrieval augmented generation combines information retrieval with language generation. A RAG system retrieves relevant documents and provides them as context to a language model.
[doc_3] Hallucination occurs when a language model generates information that is false, unsupported, or not grounded in the available evidence.

STEP 2 — PROMPT CONSTRUCTION

Answer the question using only the provided context.

Context:
Retrieval augmented generation combines information retrieval with language generation. A RAG system retrieves relevant documents and provides them as context to a language model.

Hallucination occurs when a language model generates information that is false, unsupported, or not grounded in the available evidence.

Question:
What is retrieval augmented generation?

Answer:


STEP 3 — GENERATION
combines information retrieval with language generation


In [37]:
failure_1_isolation = isolate_components(
    "What is retrieval augmented generation?",
    retriever=bad_retrieve
)

QUERY
What is retrieval augmented generation?

STEP 1 — RETRIEVAL
[doc_2] Chunking divides large documents into smaller pieces. Good chunking helps a RAG system retrieve relevant information efficiently and reduces irrelevant context.
[doc_3] Hallucination occurs when a language model generates information that is false, unsupported, or not grounded in the available evidence.

STEP 2 — PROMPT CONSTRUCTION

Answer the question using only the provided context.

Context:
Chunking divides large documents into smaller pieces. Good chunking helps a RAG system retrieve relevant information efficiently and reduces irrelevant context.

Hallucination occurs when a language model generates information that is false, unsupported, or not grounded in the available evidence.

Question:
What is retrieval augmented generation?

Answer:


STEP 3 — GENERATION
Good chunking helps a RAG system retrieve relevant information efficiently and reduces irrelevant context


In [38]:
failure_2_isolation = isolate_components(
    "What is hallucination in a large language model?"
)

QUERY
What is hallucination in a large language model?

STEP 1 — RETRIEVAL
[doc_3] Hallucination occurs when a language model generates information that is false, unsupported, or not grounded in the available evidence.
[doc_1] Retrieval augmented generation combines information retrieval with language generation. A RAG system retrieves relevant documents and provides them as context to a language model.

STEP 2 — PROMPT CONSTRUCTION

Answer the question using only the provided context.

Context:
Hallucination occurs when a language model generates information that is false, unsupported, or not grounded in the available evidence.

Retrieval augmented generation combines information retrieval with language generation. A RAG system retrieves relevant documents and provides them as context to a language model.

Question:
What is hallucination in a large language model?

Answer:


STEP 3 — GENERATION
false, unsupported, or not grounded in the available evidence


In [39]:
query = "What is hallucination in a large language model?"

retrieved_docs = retrieve(query)

context = "\n\n".join(
    doc["content"] for doc in retrieved_docs
)

bad_prompt = f"""
Answer the question from your own knowledge.

Question:
{query}

Additional information:
{context}
"""

print("RETRIEVED CONTEXT:")
print(context)

print("\nBAD PROMPT:")
print(bad_prompt)

RETRIEVED CONTEXT:
Hallucination occurs when a language model generates information that is false, unsupported, or not grounded in the available evidence.

Retrieval augmented generation combines information retrieval with language generation. A RAG system retrieves relevant documents and provides them as context to a language model.

BAD PROMPT:

Answer the question from your own knowledge.

Question:
What is hallucination in a large language model?

Additional information:
Hallucination occurs when a language model generates information that is false, unsupported, or not grounded in the available evidence.

Retrieval augmented generation combines information retrieval with language generation. A RAG system retrieves relevant documents and provides them as context to a language model.



In [40]:
query = "What is chunking important for in a RAG system?"

retrieved_docs = retrieve(query)

context = "\n\n".join(
    doc["content"] for doc in retrieved_docs
)

drift_prompt = f"""
Give a detailed answer to this question.

Context:
{context}

Question:
{query}

Include additional useful information even if it is not explicitly present in the context.
"""

print("RETRIEVED CONTEXT:")
print(context)

print("\nDRIFT PROMPT:")
print(drift_prompt)

RETRIEVED CONTEXT:
Chunking divides large documents into smaller pieces. Good chunking helps a RAG system retrieve relevant information efficiently and reduces irrelevant context.

Hallucination occurs when a language model generates information that is false, unsupported, or not grounded in the available evidence.

DRIFT PROMPT:

Give a detailed answer to this question.

Context:
Chunking divides large documents into smaller pieces. Good chunking helps a RAG system retrieve relevant information efficiently and reduces irrelevant context.

Hallucination occurs when a language model generates information that is false, unsupported, or not grounded in the available evidence.

Question:
What is chunking important for in a RAG system?

Include additional useful information even if it is not explicitly present in the context.



In [41]:
from langsmith import traceable

@traceable(
    name="retrieval",
    project_name="day-33-ai-debugging"
)
def traced_retrieve(query):
    return retrieve(query)

In [42]:
@traceable(
    name="prompt_construction",
    project_name="day-33-ai-debugging"
)
def traced_prompt(context, query):
    return f"""
Answer the question using only the provided context.

Context:
{context}

Question:
{query}

Answer:
"""

In [43]:
@traceable(
    name="generation",
    project_name="day-33-ai-debugging"
)
def traced_generation(prompt_text):
    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [44]:
@traceable(
    name="Day33_Debuggable_RAG",
    project_name="day-33-ai-debugging"
)
def debug_rag_pipeline(query):
    retrieved_docs = traced_retrieve(query)

    context = "\n\n".join(
        doc["content"] for doc in retrieved_docs
    )

    prompt_text = traced_prompt(
        context,
        query
    )

    answer = traced_generation(
        prompt_text
    )

    return {
        "query": query,
        "retrieved_docs": retrieved_docs,
        "context": context,
        "prompt": prompt_text,
        "answer": answer
    }

In [45]:
trace_result = debug_rag_pipeline(
    "What is retrieval augmented generation?"
)

print("Answer:")
print(trace_result["answer"])

Answer:
combines information retrieval with language generation


In [46]:
@traceable(
    name="Failure_1_Bad_Retrieval",
    project_name="day-33-ai-debugging"
)
def traced_failure_1(query):
    retrieved_docs = bad_retrieve(query)

    context = "\n\n".join(
        doc["content"] for doc in retrieved_docs
    )

    prompt_text = traced_prompt(
        context,
        query
    )

    answer = traced_generation(
        prompt_text
    )

    return {
        "query": query,
        "retrieved_docs": retrieved_docs,
        "context": context,
        "answer": answer
    }

In [47]:
failure_1_trace = traced_failure_1(
    "What is retrieval augmented generation?"
)

print("Failure 1 answer:")
print(failure_1_trace["answer"])

Failure 1 answer:
Good chunking helps a RAG system retrieve relevant information efficiently and reduces irrelevant context


In [48]:
@traceable(
    name="Failure_2_Bad_Prompt",
    project_name="day-33-ai-debugging"
)
def traced_failure_2(query):
    retrieved_docs = retrieve(query)

    context = "\n\n".join(
        doc["content"] for doc in retrieved_docs
    )

    bad_prompt = f"""
Answer the question from your own knowledge.

Question:
{query}

Additional information:
{context}
"""

    answer = traced_generation(
        bad_prompt
    )

    return {
        "query": query,
        "retrieved_docs": retrieved_docs,
        "context": context,
        "prompt": bad_prompt,
        "answer": answer
    }

In [49]:
failure_2_trace = traced_failure_2(
    "What is hallucination in a large language model?"
)

print("Failure 2 answer:")
print(failure_2_trace["answer"])

Failure 2 answer:
false, unsupported, or not grounded in the available evidence


In [50]:
@traceable(
    name="Failure_3_Generation_Drift",
    project_name="day-33-ai-debugging"
)
def traced_failure_3(query):
    retrieved_docs = retrieve(query)

    context = "\n\n".join(
        doc["content"] for doc in retrieved_docs
    )

    drift_prompt = f"""
Give a detailed answer to this question.

Context:
{context}

Question:
{query}

Include additional useful information even if it is not explicitly present in the context.
"""

    answer = traced_generation(
        drift_prompt
    )

    return {
        "query": query,
        "retrieved_docs": retrieved_docs,
        "context": context,
        "prompt": drift_prompt,
        "answer": answer
    }

In [51]:
failure_3_trace = traced_failure_3(
    "What is chunking important for in a RAG system?"
)

print("Failure 3 answer:")
print(failure_3_trace["answer"])

Failure 3 answer:
retrieve relevant information efficiently and reduces irrelevant context


In [52]:
import logging
import json
import time
import uuid
from datetime import datetime

logger = logging.getLogger("day33")
logger.setLevel(logging.INFO)

logger.handlers.clear()

handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter("%(message)s"))

logger.addHandler(handler)

print("JSON logger configured")

JSON logger configured


In [53]:
def log_step(session_id, step_name, input_summary, output_summary, latency_ms):
    log_entry = {
        "timestamp": datetime.now().isoformat(),
        "session_id": session_id,
        "step_name": step_name,
        "input_summary": input_summary,
        "output_summary": output_summary,
        "latency_ms": round(latency_ms, 2)
    }

    logger.info(json.dumps(log_entry))

In [54]:
def logged_rag_pipeline(query):
    session_id = str(uuid.uuid4())

    start = time.perf_counter()

    retrieved_docs = retrieve(query)

    latency = (time.perf_counter() - start) * 1000

    log_step(
        session_id,
        "retrieval",
        query,
        [doc["id"] for doc in retrieved_docs],
        latency
    )

    start = time.perf_counter()

    context = "\n\n".join(
        doc["content"] for doc in retrieved_docs
    )

    prompt_text = f"""
Answer the question using only the provided context.

Context:
{context}

Question:
{query}

Answer:
"""

    latency = (time.perf_counter() - start) * 1000

    log_step(
        session_id,
        "prompt_construction",
        query,
        prompt_text[:200],
        latency
    )

    start = time.perf_counter()

    answer = local_generate(
        context,
        query
    )

    latency = (time.perf_counter() - start) * 1000

    log_step(
        session_id,
        "generation",
        prompt_text[:200],
        answer[:200],
        latency
    )

    return {
        "session_id": session_id,
        "query": query,
        "retrieved_docs": retrieved_docs,
        "prompt": prompt_text,
        "answer": answer
    }

In [55]:
logged_result = logged_rag_pipeline(
    "What is retrieval augmented generation?"
)

print("\nFinal Answer:")
print(logged_result["answer"])

{"timestamp": "2026-09-15T14:38:58.185698", "session_id": "1b372c05-18a3-4319-a39b-be37419ad40c", "step_name": "retrieval", "input_summary": "What is retrieval augmented generation?", "output_summary": ["doc_1", "doc_3"], "latency_ms": 0.06}
INFO:day33:{"timestamp": "2026-09-15T14:38:58.185698", "session_id": "1b372c05-18a3-4319-a39b-be37419ad40c", "step_name": "retrieval", "input_summary": "What is retrieval augmented generation?", "output_summary": ["doc_1", "doc_3"], "latency_ms": 0.06}
{"timestamp": "2026-09-15T14:38:58.191126", "session_id": "1b372c05-18a3-4319-a39b-be37419ad40c", "step_name": "prompt_construction", "input_summary": "What is retrieval augmented generation?", "output_summary": "\nAnswer the question using only the provided context.\n\nContext:\nRetrieval augmented generation combines information retrieval with language generation. A RAG system retrieves relevant documents and pr", "latency_ms": 0.01}
INFO:day33:{"timestamp": "2026-09-15T14:38:58.191126", "session_i


Final Answer:
combines information retrieval with language generation


In [56]:
def improved_retrieve(query, k=2):
    query_words = set(
        word.lower().strip(".,?!")
        for word in query.split()
        if len(word) > 2
    )

    scored_documents = []

    for doc in documents:
        doc_words = set(
            word.lower().strip(".,?!")
            for word in doc["content"].split()
        )

        score = len(query_words.intersection(doc_words))

        scored_documents.append(
            (score, doc)
        )

    scored_documents.sort(
        key=lambda x: x[0],
        reverse=True
    )

    return [
        doc for score, doc in scored_documents[:k]
        if score > 0
    ]

In [57]:
query = "What is retrieval augmented generation?"

old_results = bad_retrieve(query)
new_results = improved_retrieve(query)

print("Old retrieval:")
for doc in old_results:
    print("-", doc["id"])

print("\nImproved retrieval:")
for doc in new_results:
    print("-", doc["id"])

Old retrieval:
- doc_2
- doc_3

Improved retrieval:
- doc_1


In [58]:
def fixed_failure_1_pipeline(query):
    retrieved_docs = improved_retrieve(query)

    context = "\n\n".join(
        doc["content"] for doc in retrieved_docs
    )

    answer = local_generate(
        context,
        query
    )

    return {
        "query": query,
        "retrieved_docs": retrieved_docs,
        "context": context,
        "answer": answer
    }

In [59]:
fixed_1 = fixed_failure_1_pipeline(
    "What is retrieval augmented generation?"
)

print("Retrieved documents:")

for doc in fixed_1["retrieved_docs"]:
    print("-", doc["id"])

print("\nFixed answer:")
print(fixed_1["answer"])

Retrieved documents:
- doc_1

Fixed answer:
combines information retrieval with language generation


In [60]:
def grounded_generate(context, question):
    prompt = f"""
You are a grounded question-answering system.

Use ONLY the information provided in the context.
Do not use outside knowledge.
Do not add facts that are not supported by the context.
If the answer cannot be found in the context, say:
"Information not available in the provided context."

Context:
{context}

Question:
{question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [61]:
def fixed_failure_2_pipeline(query):
    retrieved_docs = retrieve(query)

    context = "\n\n".join(
        doc["content"] for doc in retrieved_docs
    )

    answer = grounded_generate(
        context,
        query
    )

    return {
        "query": query,
        "retrieved_docs": retrieved_docs,
        "context": context,
        "answer": answer
    }

In [62]:
fixed_2 = fixed_failure_2_pipeline(
    "What is hallucination in a large language model?"
)

print("Retrieved documents:")

for doc in fixed_2["retrieved_docs"]:
    print("-", doc["id"])

print("\nFixed answer:")
print(fixed_2["answer"])

Retrieved documents:
- doc_3
- doc_1

Fixed answer:
information that is false, unsupported, or not grounded in the available evidence


In [63]:
import re

def get_words(text):
    return set(
        word.lower()
        for word in re.findall(r"[a-zA-Z]+", text)
        if len(word) > 2
    )

def evaluate_answer(answer, reference, context):
    answer_words = get_words(answer)
    reference_words = get_words(reference)
    context_words = get_words(context)

    reference_overlap = len(answer_words & reference_words) / max(len(reference_words), 1)
    context_overlap = len(answer_words & context_words) / max(len(answer_words), 1)

    if reference_overlap >= 0.70:
        correctness = 4
    elif reference_overlap >= 0.50:
        correctness = 3
    elif reference_overlap >= 0.25:
        correctness = 2
    elif reference_overlap > 0:
        correctness = 1
    else:
        correctness = 0

    if context_overlap >= 0.70:
        groundedness = 4
    elif context_overlap >= 0.50:
        groundedness = 3
    elif context_overlap >= 0.25:
        groundedness = 2
    elif context_overlap > 0:
        groundedness = 1
    else:
        groundedness = 0

    return {
        "correctness": correctness,
        "groundedness": groundedness
    }

print("Evaluation scorer ready")

Evaluation scorer ready


In [64]:
failure_pipelines = [
    ("failure_1", traced_failure_1, evaluation_cases[0]),
    ("failure_2", traced_failure_2, evaluation_cases[1]),
    ("failure_3", traced_failure_3, evaluation_cases[2])
]

before_results = []

for failure_id, pipeline, case in failure_pipelines:
    result = pipeline(case["query"])

    scores = evaluate_answer(
        result["answer"],
        case["reference"],
        result["context"]
    )

    before_results.append({
        "id": failure_id,
        "query": case["query"],
        "correctness": scores["correctness"],
        "groundedness": scores["groundedness"],
        "answer": result["answer"]
    })

print("BEFORE FIX")
print("=" * 60)

for item in before_results:
    print(item["id"])
    print("Correctness:", item["correctness"])
    print("Groundedness:", item["groundedness"])
    print("Answer:", item["answer"])
    print()

BEFORE FIX
failure_1
Correctness: 2
Groundedness: 4
Answer: Good chunking helps a RAG system retrieve relevant information efficiently and reduces irrelevant context

failure_2
Correctness: 2
Groundedness: 4
Answer: Good chunking helps a RAG system retrieve relevant information efficiently and reduces irrelevant context

failure_3
Correctness: 0
Groundedness: 4
Answer: Retrieval augmented generation



In [65]:
fixed_pipelines = [
    ("failure_1_fixed", fixed_failure_1_pipeline, evaluation_cases[0]),
    ("failure_2_fixed", fixed_failure_2_pipeline, evaluation_cases[1])
]

after_results = []

for failure_id, pipeline, case in fixed_pipelines:
    result = pipeline(case["query"])

    scores = evaluate_answer(
        result["answer"],
        case["reference"],
        result["context"]
    )

    after_results.append({
        "id": failure_id,
        "query": case["query"],
        "correctness": scores["correctness"],
        "groundedness": scores["groundedness"],
        "answer": result["answer"]
    })

print("AFTER FIX")
print("=" * 60)

for item in after_results:
    print(item["id"])
    print("Correctness:", item["correctness"])
    print("Groundedness:", item["groundedness"])
    print("Answer:", item["answer"])
    print()

AFTER FIX
failure_1_fixed
Correctness: 2
Groundedness: 4
Answer: combines information retrieval with language generation

failure_2_fixed
Correctness: 0
Groundedness: 0
Answer: a).



In [66]:
print("=" * 75)
print("DAY 33 — BEFORE vs AFTER")
print("=" * 75)

for before, after in zip(before_results[:2], after_results):
    print(f"\n{before['id']}")
    print("-" * 50)

    print(
        f"Correctness: "
        f"{before['correctness']} -> {after['correctness']}"
    )

    print(
        f"Groundedness: "
        f"{before['groundedness']} -> {after['groundedness']}"
    )

    print(
        "Status:",
        "FIX IMPROVED" if (
            after["correctness"] >= before["correctness"] and
            after["groundedness"] >= before["groundedness"]
        ) else "NEEDS REVIEW"
    )

DAY 33 — BEFORE vs AFTER

failure_1
--------------------------------------------------
Correctness: 2 -> 2
Groundedness: 4 -> 4
Status: FIX IMPROVED

failure_2
--------------------------------------------------
Correctness: 2 -> 0
Groundedness: 4 -> 0
Status: NEEDS REVIEW


In [67]:
final_report = {
    "challenge": "ABTalks 60 Days Claude AI Challenge",
    "day": 33,
    "topic": "Debugging AI Systems Systematically",
    "failures_reproduced": 3,
    "failures_fixed": 2,
    "before": before_results,
    "after": after_results,
    "remaining_failure": "failure_3_generation_drift"
}

with open("day33_debugging_report.json", "w") as f:
    json.dump(final_report, f, indent=2)

print("Final evaluation report saved as day33_debugging_report.json")

Final evaluation report saved as day33_debugging_report.json


In [68]:
runbook = """
DAY 33 — AI SYSTEM DEBUGGING RUNBOOK

1. IDENTIFY THE FAILURE
- Run the evaluation suite.
- Find queries with low correctness or groundedness scores.
- Record the query, expected answer, actual answer, and evaluation score.

2. REPRODUCE THE FAILURE
- Run the same query again using the same pipeline configuration.
- Confirm that the failure can be reproduced.
- Save the input, retrieved documents, prompt, and generated answer.

3. INSPECT THE LANGSMITH TRACE
- Open the corresponding LangSmith trace.
- Inspect retrieval, prompt construction, and generation.
- Identify where the incorrect output first appears.

4. ISOLATE COMPONENTS
Test each component independently:
- Retrieval: check whether the correct documents are retrieved.
- Prompt construction: check whether retrieved context is correctly inserted.
- Generation: check whether the model follows the context and instructions.

5. CHECK STRUCTURED LOGS
Review JSON logs containing:
- timestamp
- session_id
- step_name
- input_summary
- output_summary
- latency_ms

6. CLASSIFY THE ROOT CAUSE
Common causes:
- Bad retrieved chunk
- Incorrect retrieval ranking
- Prompt ignoring retrieved context
- Generation drift
- Unsupported information added by the model

7. APPLY A TARGETED FIX
Possible fixes:
- Improve chunking or retrieval configuration.
- Adjust retrieval threshold.
- Strengthen the system prompt.
- Restrict generation to retrieved context.

8. RE-RUN THE FAILED QUERY
- Run the same query after applying the fix.
- Compare the new correctness and groundedness scores with the original scores.

9. CHECK FOR REGRESSION
- Re-run the evaluation suite.
- Confirm that the fix improves the target failure.
- Confirm that other test cases do not become worse.

10. DOCUMENT THE RESULT
Record:
- Original failure
- Root cause
- Fix applied
- Before score
- After score
- Regression result

DAY 33 IMPLEMENTATION SUMMARY
- 3 failures deliberately reproduced.
- LangSmith tracing added.
- Retrieval, prompt, and generation components isolated.
- Structured JSON logging implemented.
- 2 failures fixed.
- Evaluation rerun after fixes.
- Remaining generation-drift failure documented for future improvement.
"""

with open("day33_debugging_runbook.txt", "w") as f:
    f.write(runbook)

print(runbook)
print("\nRunbook saved as day33_debugging_runbook.txt")


DAY 33 — AI SYSTEM DEBUGGING RUNBOOK

1. IDENTIFY THE FAILURE
- Run the evaluation suite.
- Find queries with low correctness or groundedness scores.
- Record the query, expected answer, actual answer, and evaluation score.

2. REPRODUCE THE FAILURE
- Run the same query again using the same pipeline configuration.
- Confirm that the failure can be reproduced.
- Save the input, retrieved documents, prompt, and generated answer.

3. INSPECT THE LANGSMITH TRACE
- Open the corresponding LangSmith trace.
- Inspect retrieval, prompt construction, and generation.
- Identify where the incorrect output first appears.

4. ISOLATE COMPONENTS
Test each component independently:
- Retrieval: check whether the correct documents are retrieved.
- Prompt construction: check whether retrieved context is correctly inserted.
- Generation: check whether the model follows the context and instructions.

5. CHECK STRUCTURED LOGS
Review JSON logs containing:
- timestamp
- session_id
- step_name
- input_summary